# S08 – PyTorch: быстрый ввод (тензоры, autograd, nn.Module)

Этот мини‑ноутбук – разогрев перед основным содержимом семинара S08. Цель ноутбука – быстро дать **mental model PyTorch**:
- данные – тензоры (shape/dtype/device);
- вычисления – граф autograd;
- модели – `nn.Module` и `state_dict`;
- обучение – `loss.backward()` и `optimizer.step()`.

## Импорты и общие настройки

In [1]:
import os
import random
import numpy as np
import torch

# Повторяемость
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
print('torch:', torch.__version__)

device: cpu
torch: 2.9.0+cpu


## 1) Тензоры: shape, dtype, индексация

PyTorch в первую очередь – библиотека тензоров.

Важные атрибуты тензора:
- `shape` (размерности),
- `dtype` (тип),
- `device` (CPU/GPU),
- `requires_grad` (нужны ли градиенты).

In [2]:
x = torch.randn(3, 4)
print('x.shape:', x.shape)
print('x.dtype:', x.dtype)
print('x.device:', x.device)

# Индексация
print('x[0]:', x[0])
print('x[:, 1]:', x[:, 1])

# Broadcasting (как в NumPy)
b = torch.randn(4)
y = x + b
print('b.shape:', b.shape, 'y.shape:', y.shape)

x.shape: torch.Size([3, 4])
x.dtype: torch.float32
x.device: cpu
x[0]: tensor([0.3367, 0.1288, 0.2345, 0.2303])
x[:, 1]: tensor([ 0.1288, -0.1863,  0.2674])
b.shape: torch.Size([4]) y.shape: torch.Size([3, 4])


## 2) Device: перенос на GPU/CPU

Правило: **и данные, и модель должны быть на одном device**.

Типовая ошибка новичков: модель на GPU, а батч на CPU (или наоборот).

In [3]:
x_gpu = x.to(device)
print('x_gpu.device:', x_gpu.device)

# Можно цепочкой задавать dtype и device
z = torch.ones((2, 3), dtype=torch.float32, device=device)
print('z:', z, z.device, z.dtype)

x_gpu.device: cpu
z: tensor([[1., 1., 1.],
        [1., 1., 1.]]) cpu torch.float32


## 3) Autograd: как получаются градиенты

PyTorch строит вычислительный граф "на лету" (eager mode).

Если тензор создан с `requires_grad=True`, то после `backward()` в нём появится `grad`.

Важно: **градиенты накапливаются**, поэтому перед новой итерацией обычно делают `zero_grad()`.

In [4]:
w = torch.tensor(2.0, requires_grad=True)
t = torch.tensor(5.0)

loss = (w - t) ** 2  # (2 - 5)^2 = 9
loss.backward()
print('loss:', loss.item())
print('w.grad:', w.grad.item())

# Повторный backward без обнуления градиента -> накопление
loss2 = (w - t) ** 2
loss2.backward()
print('w.grad after 2nd backward (accumulated):', w.grad.item())

# Обнуляем градиент вручную
w.grad.zero_()
print('w.grad after zero_():', w.grad.item())

loss: 9.0
w.grad: -6.0
w.grad after 2nd backward (accumulated): -12.0
w.grad after zero_(): 0.0


## 4) no_grad: когда градиенты не нужны

На **валидации и инференсе** градиенты не нужны – иначе вы тратите память и время.

Используйте `torch.no_grad()` или `torch.inference_mode()`.

In [5]:
a = torch.randn(2, 2, requires_grad=True)
with torch.no_grad():
    b = a * 2
print('b.requires_grad:', b.requires_grad)

b.requires_grad: False


## 5) nn.Module: минимальная модель и параметры

В PyTorch модель – это класс, наследующий `nn.Module`.

Ключевые вещи:
- слои объявляются в `__init__`,
- вычисления – в `forward`,
- параметры доступны через `model.parameters()` и `state_dict()`.

In [6]:
import torch.nn as nn

class TinyNet(nn.Module):
    def __init__(self, in_features=4, hidden=8, out_features=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden),
            nn.ReLU(),
            nn.Linear(hidden, out_features)
        )

    def forward(self, x):
        return self.net(x)

model = TinyNet().to(device)
print(model)

# Сколько параметров?
n_params = sum(p.numel() for p in model.parameters())
print('params:', n_params)

TinyNet(
  (net): Sequential(
    (0): Linear(in_features=4, out_features=8, bias=True)
    (1): ReLU()
    (2): Linear(in_features=8, out_features=3, bias=True)
  )
)
params: 67


## 6) Один шаг обучения: loss.backward и optimizer.step

Покажем самый маленький кусок обучения:
1) forward
2) loss
3) backward
4) step

Замечание: перед backward обычно делаем `optimizer.zero_grad()`.

In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

# Игрушечный батч: 5 объектов, 4 признака
xb = torch.randn(5, 4, device=device)
yb = torch.randint(0, 3, (5,), device=device)  # метки классов 0..2

model.train()
optimizer.zero_grad()
logits = model(xb)
loss = criterion(logits, yb)
loss.backward()
optimizer.step()

print('loss:', float(loss))

loss: 1.212571620941162


/tmp/ipython-input-407342005.py:15: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print('loss:', float(loss))


## 7) Dataset/DataLoader: как подаём данные батчами

`Dataset` отвечает за доступ к объектам, `DataLoader` – за батчи, перемешивание, параллельную загрузку.

In [8]:
from torch.utils.data import Dataset, DataLoader

class ToyDataset(Dataset):
    def __init__(self, n=1000, in_features=4, n_classes=3):
        self.X = torch.randn(n, in_features)
        self.y = torch.randint(0, n_classes, (n,))
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

ds = ToyDataset(n=32)
loader = DataLoader(ds, batch_size=8, shuffle=True)

xb, yb = next(iter(loader))
print('xb.shape:', xb.shape, 'yb.shape:', yb.shape)

xb.shape: torch.Size([8, 4]) yb.shape: torch.Size([8])


## 8) Сохранение и загрузка: state_dict

В PyTorch обычно сохраняют **веса** (а не весь объект модели):
- `torch.save(model.state_dict(), path)`
- `model.load_state_dict(torch.load(path))`

In [9]:
tmp_path = 'tinynet_state_dict.pt'
torch.save(model.state_dict(), tmp_path)
print('saved to', tmp_path, 'size:', os.path.getsize(tmp_path), 'bytes')

# Загрузка в новый экземпляр
model2 = TinyNet().to(device)
state = torch.load(tmp_path, map_location=device)
model2.load_state_dict(state)
model2.eval()

with torch.no_grad():
    out1 = model(xb.to(device))
    out2 = model2(xb.to(device))
print('max |out1-out2|:', (out1 - out2).abs().max().item())

saved to tinynet_state_dict.pt size: 2827 bytes
max |out1-out2|: 0.0


## 9) Чек‑лист типовых ошибок

1) `CrossEntropyLoss` ждёт **logits**, не `softmax`, и `y` должен быть `LongTensor` с индексами классов.

2) `model.train()` и `model.eval()` меняют поведение Dropout, BatchNorm.

3) Градиенты накапливаются – нужен `optimizer.zero_grad()`.

4) Данные и модель должны быть на одном `device`.

5) На валидации используйте `torch.no_grad()`.